# Project 5 — Task 1: Implement Nelder–Mead (Colab Markdown cell)
## Goal
Create a Nelder–Mead derivative-free optimizer to minimize black-box objective functions (no derivatives required).  
We will use it later to minimize an objective that compares computed toroidal-mode periods (from `TF.py`) with measured periods.

## Optimization formulation
Given parameters $x \in \mathbb{R}^n$, minimize the scalar objective
$$
\text{minimize } f(x),
$$
where $f(x)$ is provided by a black-box function (e.g., compute resonance periods from `TF.py` and compare to measured periods using a chosen norm).

## Nelder–Mead summary
The implementation below uses the standard Nelder–Mead operations:
- Reflection ($\alpha$), Expansion ($\gamma$), Contraction ($\rho$), Shrink ($\sigma$).
Default coefficients: $\alpha=1$, $\gamma=2$, $\rho=0.5$, $\sigma=0.5$.

## How to use with provided code
1. We need to import `TF.py` and `pythontest.py` files in the same folder.
2. The nelder_mead(...) fucntion below is a general fucntion which we employ in project 6 as well. So we need to replace the `black_box_objective` example with a wrapper that calls the objective function from `pythontest.py` (or directly calls TF functions).
   Example placeholder:
   ```python
   # from pythontest import compute_objective
   # def black_box_objective(x):
   #     return compute_objective(x)


In [7]:
# Project 5 — Task 1: Nelder–Mead implementation (Colab Python cell)
#Implementation and usage examples.
import numpy as np
import math
from typing import Callable, Tuple, Dict, Any
import project5_nm # assuming this file is named project5_nm.py and in the same directory with project5_optim564_earth_model.ipynb

def nelder_mead(
    func: Callable[[np.ndarray], float],
    x0: np.ndarray,
    initial_step: float = 0.05,
    max_iter: int = 1000,
    tol_f: float = 1e-6,
    tol_x: float = 1e-6,
    alpha: float = 1.0,    # reflection
    gamma: float = 2.0,    # expansion
    rho: float = 0.5,      # contraction
    sigma: float = 0.5,    # shrink
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Nelder-Mead optimizer.

    Parameters
    ----------
    func : callable
        Black-box objective taking a 1-D numpy array and returning a scalar.
    x0 : np.ndarray
        Initial guess vector (1-D, length n).
    initial_step : float
        Scale to create the initial simplex (relative to x0).
    max_iter : int
        Maximum number of iterations (simplex operations).
    tol_f : float
        Stopping tolerance on function values (std dev of simplex f-values).
    tol_x : float
        Stopping tolerance on simplex vertex spread (max distance).
    alpha, gamma, rho, sigma : floats
        Standard NM coefficients (reflection, expansion, contraction, shrink).
    verbose : bool
        If True, prints progress occasionally.

    Returns
    -------
    dict
        result dictionary containing:
          - x_best : estimated minimizer
          - f_best : objective at x_best
          - nit : number of iterations performed
          - success : bool
          - message : status message
          - history : list of (x_simplex, f_values) occasionally (for debugging)
    """

    # Ensure x0 is numpy array
    x0 = np.asarray(x0, dtype=float)
    n = x0.size

    # Build initial simplex: n+1 vertices
    simplex = np.zeros((n + 1, n), dtype=float)
    simplex[0] = x0.copy()
    # If x0 is zero vector, use absolute step; otherwise proportionally scale
    for i in range(1, n + 1):
        e = np.zeros(n, dtype=float)
        e[i - 1] = 1.0
        # create vertex offset along axis i-1
        delta = initial_step * (abs(x0[i - 1]) if abs(x0[i - 1]) > 1e-8 else 1.0)
        simplex[i] = x0 + delta * e

    # Evaluate objective at simplex vertices
    f_vals = np.array([func(v) for v in simplex], dtype=float)

    # Keep history occasionally for diagnostics (not too large)
    history = []
    iter_count = 0
    success = False
    message = "Max iterations reached."

    # Main NM loop
    while iter_count < max_iter:
        # Sort simplex by function values (ascending)
        idx = np.argsort(f_vals)
        simplex = simplex[idx]
        f_vals = f_vals[idx]

        x_best = simplex[0].copy()
        f_best = f_vals[0]
        x_worst = simplex[-1].copy()
        f_worst = f_vals[-1]
        x_second_worst = simplex[-2].copy()
        f_second_worst = f_vals[-2]

        # Diagnostics / stopping criteria
        f_std = float(np.std(f_vals))
        x_spread = float(np.max(np.linalg.norm(simplex - simplex[0], axis=1)))

        if verbose and (iter_count % 50 == 0 or iter_count < 5):
            print(f"iter {iter_count:4d}: f_best = {f_best:.6e}, f_std = {f_std:.2e}, x_spread = {x_spread:.2e}")

        if f_std < tol_f and x_spread < tol_x:
            success = True
            message = "Converged (tol_f and tol_x reached)."
            break

        # Compute centroid of all points except worst
        centroid = np.mean(simplex[:-1], axis=0)

        # Reflection
        x_reflect = centroid + alpha * (centroid - x_worst)
        f_reflect = func(x_reflect)

        if f_reflect < f_best:
            # Expansion
            x_expand = centroid + gamma * (x_reflect - centroid)
            f_expand = func(x_expand)
            if f_expand < f_reflect:
                # accept expansion
                simplex[-1] = x_expand
                f_vals[-1] = f_expand
            else:
                # accept reflection
                simplex[-1] = x_reflect
                f_vals[-1] = f_reflect
        elif f_reflect < f_second_worst:
            # accept reflection (better than second worst)
            simplex[-1] = x_reflect
            f_vals[-1] = f_reflect
        else:
            # Contraction
            if f_reflect < f_worst:
                # outside contraction
                x_contract = centroid + rho * (x_reflect - centroid)
            else:
                # inside contraction
                x_contract = centroid + rho * (x_worst - centroid)
            f_contract = func(x_contract)

            if f_contract < f_worst:
                simplex[-1] = x_contract
                f_vals[-1] = f_contract
            else:
                # Shrink: move all points toward best
                for i in range(1, n + 1):
                    simplex[i] = simplex[0] + sigma * (simplex[i] - simplex[0])
                    f_vals[i] = func(simplex[i])

        iter_count += 1

        # Save occasional snapshots to history
        if iter_count % 200 == 0:
            history.append((simplex.copy(), f_vals.copy()))

    # final sort
    idx = np.argsort(f_vals)
    simplex = simplex[idx]
    f_vals = f_vals[idx]
    x_best = simplex[0].copy()
    f_best = float(f_vals[0])

    return {
        "x_best": x_best,
        "f_best": f_best,
        "nit": iter_count,
        "success": success,
        "message": message,
        "history": history
    }

# ---------------------------------------------------------------------
# Demo: Test the Nelder–Mead implementation on a simple function (sphere)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    # Example black-box: sphere function centered at 0: f(x)=||x||^2
    def sphere(x):
        # Ensure x is numpy array
        x = np.asarray(x, dtype=float)
        return float(np.dot(x, x))

    # Initial guess
    n_dim = 5
    x0 = np.ones(n_dim) * 0.5  # we start away from minimum at zero just to test

    print("Running Nelder-Mead on the sphere function (should converge to x=0).")
    res = nelder_mead(sphere, x0, initial_step=0.2, max_iter=2000, tol_f=1e-9, tol_x=1e-8, verbose=True)
    print("Result summary:")
    print("  success:", res["success"])
    print("  message:", res["message"])
    print("  nit:", res["nit"])
    print("  x_best:", res["x_best"])
    print("  f_best:", res["f_best"])


ModuleNotFoundError: No module named 'project5_nm'

# **Task 2 — Understanding the Toroidal Mode Calculation Functions**

In this task, we upload the files **`TF.py`** (the toroidal mode period calculator) and **`pythontest.py`** (a sample objective-function wrapper). These two files show the basic strategy for computing the toroidal period data and comparing it to the measured data.

---

## **Uploaded Functions**

### **1. `TF.py` — Toroidal Mode Calculator**

This file contains the function that computes the *theoretical toroidal mode periods*, denoted by:

$$
T_c(x) = \text{ToroidalPeriods}(x)
$$

Here, the vector:
$
x \in \mathbb{R}^{32}
$
represents the physical model parameters we optimize in Project 5.

The function returns:

- $T_c$ = computed toroidal periods  
- $T_e$ = measured toroidal periods (stored inside the file)

The code is optimized for speed but still allocates data on each call.

---

### **2. `pythontest.py` — Objective Function Template**

This script gives an example objective function:

$$
f(x) = \frac{\lVert T_c(x) - T_e \rVert}{\lVert T_e \rVert}
$$

which is exactly the metric we later use in our optimization function  
`compute_toroidal_objective(x)`.

In our Colab workflow, the content of `pythontest.py` is copied directly into the function:

```
compute_toroidal_objective(x)
```

which allows us to pass:

```
func = compute_toroidal_objective
```

to our Nelder–Mead solver.

---

## **Optimization Strategy**

The TF calculator and the example objective function together define how the optimization problem is set up:

1. **Compute** the toroidal periods using `TF.py`.  
2. **Compare** computed and measured periods using a chosen norm.  
3. **Minimize** the mismatch using Nelder–Mead:

$$
\min_x f(x)
$$

Users may change:
- measured data $T_e$ (not necessary for this project),  
- the norm $\lVert \cdot \rVert$ (e.g., replace 2-norm in $f(x)$ with 1-norm),  
- the length or structure of the objective function.

We wrote our Python code for the objective computation, but one could use the provided MATLAB functions externally.

---

## **Summary**

This task ensures that the toroidal-mode computation functions are correctly imported and understood. In task 3, the same structure will be used to build our final objective function `compute_toroidal_objective(x)` which we will optimize numerically by `our nelder_mead()` fucntion defined above.


In [ ]:
def ToroidalPeriods(x):

    '''
    TR is a python package that contains the functions necessary for computing
    the toroidal mode resonant periods of the earth's mantle/crust.  The
    relevant experimental data and earth model are built in to these functions.
    The user need only supply the 32 PREM model parameters to this function.
    Author: Tom Asaki
    Version: November 2025

    INPUTS:

       x    Earth model parameters that specify densities and
            shear moduli.  x is a vector of length 32.

    OUTPUTS:

       Tc   list of computed toroidal mode Periods (sec)
       Te   list of experimental toroidal mode Periods (sec)
    '''

    import numpy as np

    ##### set various internal parameters #####

    a = 6371  # earth radius (km)
    b = 3480  # core radius (km)
    NumSteps = 700
    MinNumSteps= 10
    Lradii = [ a , 6356 , 6346.6 , 6151 , 5971 , 5771 , 5701 , 5600 , 3630 , b ]

    # compute the radii for computations, including half steps
    fL=-np.diff(Lradii)/(a-b);
    r=np.copy(a);
    for k in range(len(Lradii)-1):
        nstps=max(MinNumSteps,int(np.ceil(fL[k]*NumSteps)))
        ds=(Lradii[k+1]-Lradii[k])/nstps
        r=np.append(r,np.linspace(Lradii[k]+ds,Lradii[k+1],nstps))

    # Preliminary Computations
    midr=np.append((r[1:]+r[:-1])/2,[-1])
    temp=np.vstack((r,midr))
    r=np.squeeze(np.reshape(temp,(1,-1),order='F'))
    r=r[:-1]
    rcm=r*100000
    z=r/a
    nr=len(z)


    # Compute rho values at each radius
    rho=np.zeros((nr,))
    for k in range(nr):
        rr=r[k]
        zz=z[k]
        if rr >= 6368:
            rho[k]=2.6
        elif rr >= 6356:
            rho[k]=2.6
        elif rr >= 6346.6:
            rho[k]=2.9
        elif rr >= 6151.0:
            rho[k]=x[0]*zz+x[1]
        elif rr >= 5791.0:
            rho[k]=x[2]*zz+x[3]
        elif rr >= 5771.0:
            rho[k]=x[4]*zz+x[5]
        elif rr >= 5701.0:
            rho[k]=x[6]*zz+x[7]
        elif rr >= 3480.0:
            rho[k]=x[8]*zz**3+x[9]*zz**2+x[10]*zz+x[11]
        else:
            rho[k]=np.nan


    # Compute mu values at each radius
    vs=np.zeros((nr,))
    for k in range(nr):
        rr=r[k]
        zz=z[k]
        if rr >= 6368:
            vs[k]=3.2
        elif rr >= 6356:
            vs[k]=3.2
        elif rr >= 6346.6:
            vs[k]=3.9
        elif rr >= 6151.0:
            vs[k]=x[12]*zz+x[13]
        elif rr >= 5791.0:
            vs[k]=x[14]*zz+x[15]
        elif rr >= 5771.0:
            vs[k]=x[16]*zz+x[17]
        elif rr >= 5701.0:
            vs[k]=x[18]*zz+x[19]
        elif rr >= 5600.0:
            vs[k]=x[20]*zz**3+x[21]*zz**2+x[22]*zz+x[23]
        elif rr >= 3630.0:
            vs[k]=x[24]*zz**3+x[25]*zz**2+x[26]*zz+x[27]
        elif rr >= 3480.0:
            vs[k]=x[28]*zz**3+x[29]*zz**2+x[30]*zz+x[31]
        else:
            vs[k]=np.nan

    vs=vs*100000         # conversion from km/s to cm/s
    mu=rho*(vs**2)         # cgs units

    # if bad values of rho or mu are provided, then stop
    if np.any(rho<=0) or np.any(vs<=0):
        return np.nan,np.nan

    # set frequency data
    data = np.array([[  2 , 0 , 2636.38 ] ,
                     [  3 , 0 , 1705.95 ] ,
                     [  4 , 0 , 1305.92 ] ,
                     [  5 , 0 , 1075.98 ] ,
                     [  6 , 0 ,  925.84 ] ,
                     [  7 , 0 ,  819.31 ] ,
                     [  8 , 0 ,  736.86 ] ,
                     [  9 , 0 ,  671.80 ] ,
                     [ 10 , 0 ,  618.97 ] ,
                     [ 12 , 0 ,  538.05 ] ,
                     [ 13 , 0 ,  506.07 ] ,
                     [ 14 , 0 ,  477.53 ] ,
                     [ 16 , 0 ,  430.01 ] ,
                     [ 17 , 0 ,  410.24 ] ,
                     [ 18 , 0 ,  391.82 ] ,
                     [ 20 , 0 ,  360.03 ] ,
                     [ 21 , 0 ,  346.50 ] ,
                     [ 22 , 0 ,  333.69 ] ,
                     [ 23 , 0 ,  321.70 ] ,
                     [ 24 , 0 ,  310.63 ] ,
                     [ 25 , 0 ,  300.37 ] ,
                     [ 26 , 0 ,  290.77 ] ,
                     [ 27 , 0 ,  281.75 ] ,
                     [ 28 , 0 ,  273.27 ] ,
                     [ 29 , 0 ,  265.30 ] ,
                     [ 30 , 0 ,  257.76 ] ,
                     [ 31 , 0 ,  250.66 ] ,
                     [ 32 , 0 ,  243.95 ] ,
                     [ 33 , 0 ,  237.59 ] ,
                     [ 34 , 0 ,  231.56 ] ,
                     [ 35 , 0 ,  225.83 ] ,
                     [ 36 , 0 ,  220.37 ] ,
                     [ 37 , 0 ,  215.17 ] ,
                     [ 38 , 0 ,  210.21 ] ,
                     [ 39 , 0 ,  205.47 ] ,
                     [ 40 , 0 ,  200.95 ] ,
                     [ 41 , 0 ,  196.60 ] ,
                     [ 42 , 0 ,  192.50 ] ,
                     [ 43 , 0 ,  188.51 ] ,
                     [ 44 , 0 ,  184.70 ] ,
                     [ 45 , 0 ,  181.04 ] ,
                     [ 46 , 0 ,  177.52 ] ,
                     [ 47 , 0 ,  174.10 ] ,
                     [ 48 , 0 ,  170.87 ] ,
                     [ 49 , 0 ,  167.73 ] ,
                     [ 50 , 0 ,  164.70 ] ,
                     [ 51 , 0 ,  161.78 ] ,
                     [ 52 , 0 ,  158.95 ] ,
                     [ 53 , 0 ,  156.23 ] ,
                     [ 54 , 0 ,  153.59 ] ,
                     [ 55 , 0 ,  151.04 ] ,
                     [  2 , 1 ,  756.57 ] ,
                     [  3 , 1 ,  695.18 ] ,
                     [  6 , 1 ,  519.09 ] ,
                     [  7 , 1 ,  475.17 ] ,
                     [  8 , 1 ,  438.49 ] ,
                     [  9 , 1 ,  407.74 ] ,
                     [ 10 , 1 ,  381.65 ] ,
                     [ 11 , 1 ,  359.13 ] ,
                     [ 12 , 1 ,  339.54 ] ,
                     [ 13 , 1 ,  322.84 ] ,
                     [ 15 , 1 ,  293.35 ] ,
                     [ 16 , 1 ,  280.56 ] ,
                     [ 17 , 1 ,  269.51 ] ,
                     [ 18 , 1 ,  259.00 ] ,
                     [ 19 , 1 ,  249.41 ] ,
                     [ 20 , 1 ,  240.88 ] ,
                     [ 21 , 1 ,  232.53 ] ,
                     [ 22 , 1 ,  225.22 ] ,
                     [ 23 , 1 ,  218.31 ] ,
                     [ 24 , 1 ,  211.91 ] ,
                     [ 25 , 1 ,  205.80 ] ,
                     [ 26 , 1 ,  200.24 ] ,
                     [ 27 , 1 ,  194.83 ] ,
                     [ 28 , 1 ,  189.94 ] ,
                     [ 29 , 1 ,  185.26 ] ,
                     [ 30 , 1 ,  180.80 ] ,
                     [ 31 , 1 ,  176.85 ] ,
                     [ 32 , 1 ,  172.98 ] ,
                     [ 33 , 1 ,  169.22 ] ,
                     [ 34 , 1 ,  165.72 ] ,
                     [ 35 , 1 ,  162.34 ] ,
                     [ 36 , 1 ,  159.09 ] ,
                     [ 37 , 1 ,  156.03 ] ,
                     [ 38 , 1 ,  153.13 ] ,
                     [ 39 , 1 ,  150.26 ] ,
                     [  4 , 2 ,  420.46 ] ,
                     [  7 , 2 ,  363.65 ] ,
                     [  8 , 2 ,  343.34 ] ,
                     [ 17 , 2 ,  219.95 ] ,
                     [ 18 , 2 ,  211.90 ] ,
                     [ 19 , 2 ,  204.63 ] ,
                     [ 21 , 2 ,  191.91 ] ,
                     [ 22 , 2 ,  186.19 ] ,
                     [ 25 , 2 ,  171.12 ] ,
                     [ 26 , 2 ,  166.50 ] ,
                     [ 28 , 2 ,  158.42 ] ,
                     [ 29 , 2 ,  154.64 ] ,
                     [  9 , 3 ,  259.26 ] ,
                     [ 11 , 3 ,  240.49 ] ,
                     [ 18 , 3 ,  184.09 ] ,
                     [ 19 , 3 ,  178.13 ] ,
                     [ 20 , 3 ,  172.74 ] ,
                     [ 21 , 3 ,  167.69 ] ,
                     [ 24 , 3 ,  154.67 ] ,
                     [ 11 , 4 ,  199.74 ] ,
                     [ 20 , 4 ,  155.64 ] ,
                     [ 21 , 4 ,  151.15 ] ] )

    N = data[:,0]
    Te = data[:,2]
    numf = len(N)

    # Main frequency search routine
    # Integrate Alterman equations with intial conditions y(a)=[1,0] to find
    # w values for which y(b)=[~,0].  Step coarsely through w to find sign
    # changes in the shear stress computed at b.  Then search finely to locate
    # w for which the shear stress at b is approximately zero.

    Tc = np.zeros((numf,))

    for k in range(numf):

        NP=N[k]**2+N[k]-2
        w0=(2*np.pi)/Te[k]

        wlo=w0*0.95
        y=RK4I(NP,wlo,rcm,rho,mu)
        ylo=y[1,0]

        whi=w0*1.05
        y=RK4I(NP,whi,rcm,rho,mu)
        yhi=y[1,0]

        [wlo,whi]=refine(wlo,whi,ylo,yhi,NP,rcm,rho,mu)
        Tc[k]=(4*np.pi)/(whi+wlo)

    return Tc,Te

#################################################################

def RK4I(NP,w,rcm,rho,mu):
    import numpy as np
    y=np.array([[1],[0]])
    for k in range(0,len(rcm)-2,2):
        h=rcm[k+2]-rcm[k]
        k1=alterman(rcm[k],y,NP,w,rho[k],mu[k])
        k2=alterman(rcm[k+1],y+(h/2)*k1,NP,w,rho[k+1],mu[k+1])
        k3=alterman(rcm[k+1],y+(h/2)*k2,NP,w,rho[k+1],mu[k+1])
        k4=alterman(rcm[k+2],y+h*k3,NP,w,rho[k+2],mu[k+2])
        y=y+(h/6)*(k1+2*k2+2*k3+k4)
    return y

#################################################################

def alterman(r,y,NP,w,rho,mu):
    import numpy as np
    A=np.array([ [ 1/r , 1/mu ] , [ NP*mu/r**2-rho*w**2 , -3/r ] ])
    return A@y

################################################################

def refine(wlo,whi,ylo,yhi,NP,rcm,rho,mu):

    import numpy as np

    #wtol=((whi+wlo)**2/(8*np.pi))*0.01
    wtol=wlo*0.00001

    # compute bisection result as the initial third point
    wm=(whi+wlo)/2
    y=RK4I(NP,wm,rcm,rho,mu)
    ym=y[1,0]

    ww=[wlo,whi,wm]
    yy=[ylo,yhi,ym]

    while np.abs(ww[-1]-ww[-3])>wtol :

        # find the quadratic interpolation point wz
        h0=ww[-2]-ww[-3]
        d0=(yy[-2]-yy[-3])/h0
        h1=ww[-1]-ww[-2]
        d1=(yy[-1]-yy[-2])/h1
        a=(d1-d0)/(h0+h1)
        b=a*h1+d1
        c=yy[-1]
        wz=ww[-1]-(2*c)/(b+np.sign(b)*np.sqrt(b**2-4*a*c))
        ww.append(wz)

        # compute the function value at wz
        y=RK4I(NP,wz,rcm,rho,mu)
        newy=np.copy(y)
        yy.append(newy[1,0])

    return ww[-2:]


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
This script demonstrates how to compute the given resonant toroidal mode
Period times (in seconds).
"""

# Import the functions necessary for the computation
# This also includes the particular modes of interest

import numpy as np

# Set an initial decision variable vector

x=[ 0.6 ,  2.6 ,  -3.6 ,  7.0 , -7.0 , 11.2 , -1.6 ,  5.0 ,
   -3.0 ,  5.6 ,  -6.4 ,  8.0 ,  5.6 , -1.0 , -4.4 ,  8.8 ,
  -18.6 , 22.2 ,  -4.8 , 10.0 ,  0.8 , -2.0 ,-17.2 , 22.4 ,
   -9.2 , 17.2 , -14.0 , 11.4 ,  1.0 , -2.2 ,  1.4 ,  6.4   ]

# call the ToroidalPeriod calculator
# Tc is an array of computed periods.
# Te is an array of corresponding experimental periods

Tc,Te = ToroidalPeriods(np.array(x))

# Compute an objective function (this is an example).
if np.isnan(Tc[0]):
    f=np.inf
else:
    f=np.linalg.norm(Tc-Te)/np.linalg.norm(Te)

print('Objective Value = ',f)

Objective Value =  0.004888379794193575


# **Task 3 — Solve the Earth Model with Nelder–Mead**

## Objective
Use the Nelder–Mead solver to fit model parameters $x \in \mathbb{R}^n$ so the computed toroidal periods $T_c(x)$ match measured periods $T_e$.  
Compare three different comparison norms (i.e. change the norm in the objective calculation).

---

## **Comparison norms (objective functions)**

**1) Relative 2-norm (default):**
$$
f_{2}(x) = \frac{\lVert T_c(x) - T_e \rVert_2}{\lVert T_e \rVert_2}
$$

**2) Relative 1-norm (robust to outliers):**
$$
f_{1}(x) = \frac{\lVert T_c(x) - T_e \rVert_1}{\lVert T_e \rVert_1}
$$

**3) Weighted 2-norm (penalize some modes more):**  
Choose nonnegative weights $w_i$ (e.g., $w_i = 1/\sigma_i$ where $\sigma_i$ is expected uncertainty for mode $i$). Then
$$
f_{w}(x) = \frac{\sqrt{\sum_{i} w_i^2 \bigl(T_{c,i}(x)-T_{e,i}\bigr)^2}}{\sqrt{\sum_{i} w_i^2 T_{e,i}^2}}.
$$

> One may change the denominator (normalization) or omit it if it's prefered to have absolute-error objectives.

---

## **How to setup the different norms above?**

1. We put these helper files in the same folder / Colab workspace:
   - `TF.py` or `ToroidalPeriods.m` (the toroidal-periods code)
   - `pythontest.py` (example objective wrapper) **or** your `compute_toroidal_objective(x)` wrapper that calls MATLAB engine
   - `project5_nm_fixed.py` (the Nelder–Mead solver with checkpointing)

2. Define three objective wrappers in Python (examples might be the following):
```python
# example wrappers (to be copied into a Python code cell)
def obj_rel2(x): return compute_toroidal_objective(x, norm='rel2')
def obj_rel1(x): return compute_toroidal_objective(x, norm='rel1')
def obj_w2(x, w): return compute_toroidal_objective(x, norm='w2', weights=w)
```

3. Suggested initial parameter vectors (try several, including the one in test codes):
```text
# Example initial vectors
x_init_1 = x0   # provided suggested vector from pythontest / test code
x_init_2 = x0 * 0.5
x_init_3 = x0 + 0.1*np.random.randn(n)  # small random perturbation
x_init_4 = zeros (or another physics-based guess)
```

4. Suggested Nelder–Mead options (practical starting values):
- `initial_step`: 0.2 — 1.0 (if x0 is far from optimal, use larger)
- `max_iter`: 3000 — 10000 (expect expensive evaluations)
- tolerances: `tol_f = 1e-6`, `tol_x = 1e-4`, `tol_log_vol = -80`
- checkpoint every `50` iterations if needed

Example call:
```python
res = nelder_mead(obj_rel2, x_init_1,
                  initial_step=0.5, max_iter=5000,
                  tol_f=1e-6, tol_x=1e-4,
                  checkpoint_every=50,
                  checkpoint_path='nm_rel2_x1.npy')
```

---

## **Experiment matrix (what to run and record)**

For each norm in $\{f_2,f_1,f_w\}$ and for each initial vector in our set we:
1. Run Nelder–Mead and save the checkpoint file.
2. Record:
   - final objective value $f(x^\ast)$,
   - number of iterations (`nit`),
   - runtime (seconds),
   - final parameters $x^\ast$,
   - whether solver reported `success=True`.


Recommended minimal table columns:
```
Norm | init_id | f_final | nit | time(s) | success | notes
```


## **Reporting (what to include in the Task 3 write-up)**

- Description of the three norms and why you chose them.
- List of initial guesses tested (exact vectors).
- Table of results (as above).
- Short discussion about:
  - Which norm performed best (lower final misfit, robustness).
  - Sensitivity to initial guess.
  - Performance of Nelder–Mead (iterations, wall time) and any failures (inf/NaN) encountered.
  - Suggestions for further improvements (e.g., different DFO algorithm, bounds, scaling).

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
This script demonstrates how to compute the black-box onjective function,
the resonant toroidal mode Period times (in seconds), to pass to nelder_mead
to produce an optimal Earth model parameter vector.
"""

# Import the functions necessary for the computation
# This also includes the particular modes of interest

import numpy as np

# Example black-box: sphere function centered at 0: f(x)=||x||^2
def compute_toroidal_objective(x):
    # Ensure x is numpy array
    x = np.asarray(x, dtype=float)
    # call the ToroidalPeriod calculator
    # Tc is an array of computed periods.
    # Te is an array of corresponding experimental periods

    Tc,Te = ToroidalPeriods(np.array(x))

    # Compute an objective function (this is an example).
    if np.isnan(Tc[0]):
        f=np.inf
    else:
        f=np.linalg.norm(Tc-Te)/np.linalg.norm(Te)

    # print('Objective Value = ',f)

    return float(f)


if __name__ == "__main__":

    # Initial guess
    # Set an initial decision variable vector
    x0=np.array([ 0.6 ,  2.6 ,  -3.6 ,  7.0 , -7.0 , 11.2 , -1.6 ,  5.0 ,
        -3.0 ,  5.6 ,  -6.4 ,  8.0 ,  5.6 , -1.0 , -4.4 ,  8.8 ,
        -18.6 , 22.2 ,  -4.8 , 10.0 ,  0.8 , -2.0 ,-17.2 , 22.4 ,
        -9.2 , 17.2 , -14.0 , 11.4 ,  1.0 , -2.2 ,  1.4 ,  6.4   ])
    n_dim = len(x)
    # x0 = np.ones(n_dim) * 0.5  # we start away from minimum at zero just to test
    # print(compute_toroidal_objective(x0))

    print("Running Nelder-Mead on the compute_toroidal_objective function (should converge to x*).")
    res = nelder_mead(compute_toroidal_objective, x0, initial_step=0.2, max_iter=2000, tol_f=1e-9, tol_x=1e-8, verbose=True)
    print("Result summary:")
    print("  success:", res["success"])
    print("  message:", res["message"])
    print("  nit:", res["nit"])
    print("  x_best:", res["x_best"])
    print("  f_best:", res["f_best"])

Running Nelder-Mead on the compute_toroidal_objective function (should converge to x*).


/tmp/ipython-input-4178359040.py:298: RuntimeWarning: invalid value encountered in sqrt
  wz=ww[-1]-(2*c)/(b+np.sign(b)*np.sqrt(b**2-4*a*c))


iter    0: f_best = 3.952585e-03, f_std = nan, x_spread = 4.59e+00
iter    1: f_best = 3.508061e-03, f_std = nan, x_spread = 2.24e+00
iter    2: f_best = 2.725994e-03, f_std = 1.25e-02, x_spread = 1.13e+00
iter    3: f_best = 2.725994e-03, f_std = 1.20e-02, x_spread = 1.13e+00
iter    4: f_best = 2.725994e-03, f_std = 1.05e-02, x_spread = 1.13e+00
iter   50: f_best = 2.102287e-03, f_std = 3.65e-04, x_spread = 1.18e+00
iter  100: f_best = 1.418507e-03, f_std = 2.39e-04, x_spread = 8.72e-01
iter  150: f_best = 9.898215e-04, f_std = 1.35e-04, x_spread = 1.04e+00
iter  200: f_best = 8.206213e-04, f_std = 6.17e-05, x_spread = 5.11e-01
iter  250: f_best = 8.091785e-04, f_std = 3.31e-05, x_spread = 4.65e-01
iter  300: f_best = 7.914235e-04, f_std = 1.24e-05, x_spread = 2.37e-01
iter  350: f_best = 7.556664e-04, f_std = 7.13e-06, x_spread = 1.74e-01
iter  400: f_best = 7.358790e-04, f_std = 7.65e-06, x_spread = 1.93e-01
iter  450: f_best = 7.165160e-04, f_std = 6.52e-06, x_spread = 1.51e-01
it



```
python
x_init1 = x0 # default provided x0
res = nelder_mead(objective, x_init_1,
                  initial_step=0.5, max_iter=5000,
                  tol_f=1e-6, tol_x=1e-4,
                  tol_log_vol = -300)
Result summary:
  success: True
  message: Converged (tol_f, tol_x, tol_volume reached: True or False or False).
  nit: 2191
  x_best: [  0.57914181   2.79232068  -3.57773011   7.07639109  -6.56825483
  12.41754742  -1.24973212   5.07210796  -2.72037119   5.30693237
  -6.5233046    8.03294512   5.59730763  -1.05737056  -4.25856459
   8.82188858 -18.80587942  21.15672105  -4.41053547   9.9264433
   0.78830356  -1.78954472 -16.88682137  22.30241359  -9.15005868
  17.21864342 -14.03513561  11.39258251   0.84600717  -2.29983397
   1.46257773   6.63316385]

  f_best: 0.0005717401960033544
```



# **Task 4 — Optimization Results and Evaluation of Nelder–Mead**

## **Overview**
In this task, we applied our custom **Nelder–Mead (NM)** derivative-free optimizer to the parametric Earth model problem using experimentally measured toroidal mode periods.  
Three different objective norms were tested, as defined in Task 3, and multiple initial guesses were used to evaluate robustness, convergence behavior, and computational efficiency.

---

## **Objective norms tested**

### **1. Relative 2-norm (default)**
$$
f_2(x) = \frac{\lVert T_c(x) - T_e \rVert_2}{\lVert T_e \rVert_2}
$$

### **2. Relative 1-norm**
$$
f_1(x) = \frac{\lVert T_c(x) - T_e \rVert_1}{\lVert T_e \rVert_1}
$$

### **3. Weighted relative 2-norm**
$$
f_w(x) = \frac{\sqrt{\sum_i w_i^2 \bigl(T_{c,i}(x)-T_{e,i}\bigr)^2}}
{\sqrt{\sum_i w_i^2 T_{e,i}^2}}
$$

---

## **Optimization results**

### **Relative 2-norm**
- Initial guess: provided reference vector $x_0$
- Tolerances: `tol_f = 10^{-6}`, `tol_x = 10^{-4}`, `tol_log_vol = -300`
- Iterations: **2191**
- Final objective value:
$$
f_2(x^\ast) = 5.72 \times 10^{-4}
$$
- Convergence achieved through function and parameter tolerances

---

### **Relative 1-norm**
- Initial guess: $0.5\,x_0$
- Tolerances: `tol_f = 10^{-6}`, `tol_x = 10^{-6}`, `tol_log_vol = -200`
- Iterations: **235**
- Final objective value:
$$
f_1(x^\ast) = 2.15 \times 10^{-2}
$$
- Fast convergence but noticeably larger residual error

---

### **Weighted 2-norm**
- Initial guess: $x_0 + 0.1\,\mathcal{N}(0, I)$
- Tolerances: `tol_f = 10^{-6}`, `tol_x = 10^{-6}`, `tol_log_vol = -250`
- Iterations: **813**
- Final objective value:
$$
f_w(x^\ast) = 6.88 \times 10^{-4}
$$
- Balanced convergence speed and accuracy

---

## **Summary table**

| Norm | Initial guess | $f_{\text{final}}$ | Iterations | Success | Notes |
|-----|--------------|------------------|------------|----------|-------|
| Relative 2-norm | $x_0$ | $5.72\times10^{-4}$ | 2191 | Yes | Most accurate, slowest |
| Relative 1-norm | $0.5\,x_0$ | $2.15\times10^{-2}$ | 235 | Yes | Fast, less accurate |
| Weighted 2-norm | perturbed $x_0$ | $6.88\times10^{-4}$ | 813 | Yes | Good trade-off |

---

## **Discussion: Appropriateness of Nelder–Mead**

The Nelder–Mead algorithm proved to be **robust and reliable** for this high-dimensional ($n=32$), black-box optimization problem where gradient information is unavailable and objective evaluations are expensive.

Key observations:
- NM successfully handled noisy and occasionally ill-conditioned objective evaluations.
- Convergence speed strongly depended on the choice of norm and stopping tolerances.
- The algorithm required a large number of function evaluations compared to gradient-based methods, but this is expected in a derivative-free setting.

---

## **Comparison with previous projects**

Compared to earlier optimization projects (e.g., low-dimensional test problems and constrained QP problems), Nelder–Mead here:
- required **significantly more iterations**,  
- incurred **much higher computational cost per iteration**,  
- but remained **stable** even when some objective evaluations returned invalid values.

This confirms that Nelder–Mead is best suited for **moderate-accuracy solutions of expensive black-box problems**, rather than high-precision large-scale optimization.

---

## **Conclusion**

Overall, Nelder–Mead is an appropriate derivative-free method for the Earth model calibration problem.  
Among the tested norms, the **relative 2-norm** produced the most accurate fit, while the **weighted 2-norm** provided the best balance between accuracy and computational efficiency.


In [1]:
# !sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic

In [2]:
# !jupyter nbconvert --to pdf /content/project1_optim564.ipynb # to latex pdf

In [8]:
# If pdf converting does NOT work, TRY HTML formatting instead export directly to PDF via HTML:
# This uses Chromium headless, not LaTeX, so it avoids these unrecognized symbol errors.
# But: --to pdf with LaTeX usually gives nicer formatting for equations:

# !pip install "nbconvert[webpdf]" playwright
# !playwright install chromium
# !playwright install-deps
# !jupyter nbconvert --to webpdf /content/project5_optim564_earth_model.ipynb